# Transformer

Собираем Transformer, который работает типа как чат-бот:

```text
вопрос пользователя
        -
      Encoder
        -
  Cross Attention
        -
      Decoder
        -
сгенерированный ответ
```

Используется датасет Hugging Face:

```python
Den4ikAI/russian_dialogues
```

Готовые pretrained-чат-боты не используются: Transformer собирается
из собственных слоёв `Encoder`, `Decoder`, `MultiHeadAttention`,
`FeedForward`, positional encoding — в стиле урока.

## Особенность датасета

В датасете есть:

```text
question
answer
relevance
```

`relevance = 1` означает связанную пару вопрос–ответ.  
`relevance = 0` — специально созданный нерелевантный ответ.

Поэтому для чат-бота обучаемся **только на relevance == 1**.

1. Подготавливаем русский диалоговый датасет.
2. Обучаем собственный WordPiece-токенизатор.
3. Собираем Transformer Encoder–Decoder.
4. Сравниваем несколько наборов гиперпараметров.
5. Выбираем лучшую конфигурацию по `val_loss`.
6. Дообучаем лучший Transformer.
7. Реализуем автогенерацию ответа.
8. Проверяем ответы на новых вопросах.
9. Считаем chrF на части тестовой выборки.
10. Формулируем выводы.

Установка библиотек

In [ ]:
%pip install -q -U datasets tokenizers sacrebleu

Импорт библиотек

In [ ]:
# Импорты
import gc
import math
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from datasets import load_dataset
from tokenizers import BertWordPieceTokenizer
import sacrebleu


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print(
        "Предупреждение: GPU не найден. "
        "Код запустится и на CPU, но обучение будет существенно тяжелее по нагрузке и дольше по времени."
    )

## Загрузка датасета

Полный датасет содержит миллионы строк, поэтому загрузка в колабу будет весь день идти, использую:
`streaming=True`.

Читаем строки постепенно и останавливаемся после набора нужного количества
качественных пар. Поэтому не требуется загружать весь датасет в память.

In [ ]:
# Загрузка датасета
DATASET_NAME = "Den4ikAI/russian_dialogues"

# Размер подготовленной выборки.
# Если GPU/время позволяют, можно увеличить до 120000–200000.
MAX_PAIRS = 80000

# Убираем слишком длинные сообщения:
# маленькому Transformer проще учиться на компактных диалогах.
MAX_QUESTION_WORDS = 28
MAX_ANSWER_WORDS = 32


dataset_stream = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
)

# Перемешиваем поток через буфер, чтобы не брать только начало датасета.
dataset_stream = dataset_stream.shuffle(
    seed=SEED,
    buffer_size=20000,
)

print(dataset_stream)

## Очистка и фильтрация

Для генерации ответа нужны только положительные пары:

```python
relevance == 1
```

Также удаляем пустые и слишком длинные реплики.

In [ ]:
# Очистка текста
def clean_text(text):
    text = str(text).lower()
    text = text.replace("ё", "е")

    # Убираем лишние пробелы.
    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


pairs = []


for row in dataset_stream:

    if int(row["relevance"]) != 1:
        continue

    question = clean_text(
        row["question"]
    )

    answer = clean_text(
        row["answer"]
    )

    if not question or not answer:
        continue

    q_words = len(
        question.split()
    )

    a_words = len(
        answer.split()
    )

    # Слишком короткие однословные пары мало помогают,
    # а очень длинные увеличивают вычисления.
    if not (
        2 <= q_words <= MAX_QUESTION_WORDS
    ):
        continue

    if not (
        2 <= a_words <= MAX_ANSWER_WORDS
    ):
        continue

    pairs.append(
        (
            question,
            answer,
        )
    )

    if len(pairs) >= MAX_PAIRS:
        break


pairs_df = pd.DataFrame(
    pairs,
    columns=[
        "question",
        "answer",
    ],
)

print(
    "Подготовлено релевантных пар:",
    len(pairs_df),
)

display(
    pairs_df.head(10)
)

## Разделение выборки

Данные перемешиваем и делим:

- `80%` — train;
- `10%` — validation;
- `10%` — test.

In [ ]:
# Разделение выборки
pairs_df = (
    pairs_df
    .sample(
        frac=1.0,
        random_state=SEED,
    )
    .reset_index(
        drop=True
    )
)


n = len(pairs_df)

train_end = int(
    n * 0.80
)

val_end = int(
    n * 0.90
)


train_df = (
    pairs_df
    .iloc[:train_end]
    .reset_index(drop=True)
)

val_df = (
    pairs_df
    .iloc[train_end:val_end]
    .reset_index(drop=True)
)

test_df = (
    pairs_df
    .iloc[val_end:]
    .reset_index(drop=True)
)


print("TRAIN:", len(train_df))
print("VAL:", len(val_df))
print("TEST:", len(test_df))

## Токенизация

Беру с урока subword-токенизацию.

Обучаем WordPiece tokenizer на русских вопросах и ответах.

Специальные токены:

```text
[PAD]   — заполнение последовательности
[UNK]   — неизвестный токен
[START] — начало
[END]   — конец
```

In [ ]:
# Токенизация
VOCAB_SIZE = 12000

# Полная длина вопроса/ответа вместе с [START] и [END].
MAX_LENGTH = 34


tokenizer = BertWordPieceTokenizer(
    lowercase=True,
    strip_accents=False,
)


def corpus_iterator():
    for question, answer in zip(
        train_df["question"],
        train_df["answer"],
    ):
        yield question
        yield answer


tokenizer.train_from_iterator(
    corpus_iterator(),
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[START]",
        "[END]",
    ],
)


PAD_ID = tokenizer.token_to_id(
    "[PAD]"
)

UNK_ID = tokenizer.token_to_id(
    "[UNK]"
)

START_ID = tokenizer.token_to_id(
    "[START]"
)

END_ID = tokenizer.token_to_id(
    "[END]"
)


ACTUAL_VOCAB_SIZE = (
    tokenizer.get_vocab_size()
)


print(
    "Размер словаря:",
    ACTUAL_VOCAB_SIZE,
)

print(
    "PAD_ID:",
    PAD_ID,
    "START_ID:",
    START_ID,
    "END_ID:",
    END_ID,
)

## Проверка токенизации

In [ ]:
# Проверка токенизации
example_text = train_df.loc[
    0,
    "question",
]

example_encoding = tokenizer.encode(
    example_text,
    add_special_tokens=False,
)

print("Текст:")
print(example_text)

print("\nТокены:")
print(example_encoding.tokens)

print("\nID:")
print(example_encoding.ids)

## Кодирование данных

Каждая строка преобразуется в:

```text
[START] токены [END] [PAD] [PAD] ...
```

In [ ]:
# Кодирование данных
# Текст -> token ids фиксированной длины
def encode_text(
    text,
    max_length=MAX_LENGTH,
):
    ids = (
        tokenizer
        .encode(
            str(text),
            add_special_tokens=False,
        )
        .ids
    )

    # Оставляем место под START и END.
    ids = ids[
        :max_length - 2
    ]

    ids = (
        [START_ID]
        + ids
        + [END_ID]
    )

    if len(ids) < max_length:
        ids += (
            [PAD_ID]
            * (
                max_length
                - len(ids)
            )
        )

    return ids


def encode_dataframe(dataframe):

    questions = np.asarray(
        [
            encode_text(text)
            for text
            in dataframe["question"]
        ],
        dtype=np.int32,
    )

    answers = np.asarray(
        [
            encode_text(text)
            for text
            in dataframe["answer"]
        ],
        dtype=np.int32,
    )

    return questions, answers


train_questions, train_answers = (
    encode_dataframe(train_df)
)

val_questions, val_answers = (
    encode_dataframe(val_df)
)

test_questions, test_answers = (
    encode_dataframe(test_df)
)


print(
    "train_questions:",
    train_questions.shape,
)

print(
    "train_answers:",
    train_answers.shape,
)

In [ ]:
# Подготовка tf.data
BATCH_SIZE = 128

AUTOTUNE = tf.data.AUTOTUNE


# question -> encoder, answer[:-1] -> decoder, answer[1:] -> target
def make_dataset(
    questions,
    answers,
    batch_size=BATCH_SIZE,
    shuffle=False,
):

    decoder_inputs = answers[
        :,
        :-1
    ]

    targets = answers[
        :,
        1:
    ]

    ds = tf.data.Dataset.from_tensor_slices(
        (
            (
                questions,
                decoder_inputs,
            ),
            targets,
        )
    )

    if shuffle:
        ds = ds.shuffle(
            min(
                len(questions),
                20000,
            ),
            seed=SEED,
        )

    ds = (
        ds
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

    return ds


train_ds = make_dataset(
    train_questions,
    train_answers,
    shuffle=True,
)

val_ds = make_dataset(
    val_questions,
    val_answers,
)

test_ds = make_dataset(
    test_questions,
    test_answers,
)

In [ ]:
# Позиционное кодирование
def positional_encoding(
    length,
    depth,
):

    if depth % 2 != 0:
        raise ValueError(
            "d_model должен быть четным"
        )

    half_depth = depth // 2

    positions = np.arange(
        length
    )[
        :,
        np.newaxis
    ]

    depths = (
        np.arange(
            half_depth
        )[
            np.newaxis,
            :
        ]
        / half_depth
    )

    angle_rates = (
        1
        / (
            10000
            ** depths
        )
    )

    angle_rads = (
        positions
        * angle_rates
    )

    pos_encoding = np.concatenate(
        [
            np.sin(angle_rads),
            np.cos(angle_rads),
        ],
        axis=-1,
    )

    return tf.cast(
        pos_encoding,
        tf.float32,
    )

## Positional Embedding

In [ ]:
# Embedding и позиция
class PositionalEmbedding(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        vocab_size,
        d_model,
        max_length,
    ):
        super().__init__()

        self.d_model = d_model

        self.embedding = (
            tf.keras.layers.Embedding(
                vocab_size,
                d_model,
            )
        )

        self.pos_encoding = (
            positional_encoding(
                max_length,
                d_model,
            )
        )

    def call(self, x):

        length = tf.shape(x)[1]

        x = self.embedding(x)

        # Масштабирование embeddings,
        # как в оригинальном Transformer.
        x *= tf.math.sqrt(
            tf.cast(
                self.d_model,
                tf.float32,
            )
        )

        x = (
            x
            + self.pos_encoding[
                tf.newaxis,
                :length,
                :
            ]
        )

        return x

## Attention-блоки

Будут использоваться:

1. **Encoder self-attention** — вопрос анализирует сам себя.
2. **Decoder causal self-attention** — ответ видит только уже сгенерированные токены.
3. **Cross-attention** — Decoder обращается к представлению вопроса из Encoder.

In [ ]:
# Self Attention Encoder
# Вопрос смотрит сам на себя
class EncoderSelfAttention(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        num_heads,
        dropout_rate,
    ):
        super().__init__()

        self.mha = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=(
                    d_model
                    // num_heads
                ),
                dropout=dropout_rate,
            )
        )

        self.add = tf.keras.layers.Add()

        self.layernorm = (
            tf.keras.layers.LayerNormalization()
        )

    def call(
        self,
        x,
        padding_mask,
        training=False,
    ):
        # MHA ожидает маску вида
        # (batch, target_length, source_length).
        # Размер 1 во второй оси разрешает broadcasting.
        attention_mask = (
            padding_mask[
                :,
                tf.newaxis,
                :
            ]
        )

        attn_output = self.mha(
            query=x,
            value=x,
            key=x,
            attention_mask=attention_mask,
            training=training,
        )

        x = self.add(
            [
                x,
                attn_output,
            ]
        )

        return self.layernorm(x)

In [ ]:
# Causal Attention Decoder
# Ответ видит только прошлые токены
class CausalSelfAttention(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        num_heads,
        dropout_rate,
    ):
        super().__init__()

        self.mha = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=(
                    d_model
                    // num_heads
                ),
                dropout=dropout_rate,
            )
        )

        self.add = tf.keras.layers.Add()

        self.layernorm = (
            tf.keras.layers.LayerNormalization()
        )

    def call(
        self,
        x,
        padding_mask,
        training=False,
    ):
        attention_mask = (
            padding_mask[
                :,
                tf.newaxis,
                :
            ]
        )

        attn_output = self.mha(
            query=x,
            value=x,
            key=x,
            attention_mask=attention_mask,

            # Decoder не должен смотреть
            # на будущие токены ответа.
            use_causal_mask=True,

            training=training,
        )

        x = self.add(
            [
                x,
                attn_output,
            ]
        )

        return self.layernorm(x)

In [ ]:
# Cross Attention
# Decoder получает контекст Encoder
class CrossAttention(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        num_heads,
        dropout_rate,
    ):
        super().__init__()

        self.mha = (
            tf.keras.layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=(
                    d_model
                    // num_heads
                ),
                dropout=dropout_rate,
            )
        )

        self.add = tf.keras.layers.Add()

        self.layernorm = (
            tf.keras.layers.LayerNormalization()
        )

    def call(
        self,
        x,
        context,
        context_mask,
        training=False,
    ):

        attention_mask = (
            context_mask[
                :,
                tf.newaxis,
                :
            ]
        )

        attn_output = self.mha(
            query=x,
            key=context,
            value=context,
            attention_mask=attention_mask,
            training=training,
        )

        x = self.add(
            [
                x,
                attn_output,
            ]
        )

        return self.layernorm(x)

In [ ]:
# Feed Forward
class FeedForward(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        dff,
        dropout_rate,
    ):
        super().__init__()

        self.seq = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(
                    dff,
                    activation="relu",
                ),

                tf.keras.layers.Dense(
                    d_model,
                ),

                tf.keras.layers.Dropout(
                    dropout_rate,
                ),
            ]
        )

        self.add = tf.keras.layers.Add()

        self.layernorm = (
            tf.keras.layers.LayerNormalization()
        )

    def call(
        self,
        x,
        training=False,
    ):

        transformed = self.seq(
            x,
            training=training,
        )

        x = self.add(
            [
                x,
                transformed,
            ]
        )

        return self.layernorm(x)

## Encoder

In [ ]:
# Encoder
class EncoderLayer(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        num_heads,
        dff,
        dropout_rate,
    ):
        super().__init__()

        self.self_attention = (
            EncoderSelfAttention(
                d_model,
                num_heads,
                dropout_rate,
            )
        )

        self.ffn = FeedForward(
            d_model,
            dff,
            dropout_rate,
        )

    def call(
        self,
        x,
        padding_mask,
        training=False,
    ):

        x = self.self_attention(
            x,
            padding_mask=padding_mask,
            training=training,
        )

        x = self.ffn(
            x,
            training=training,
        )

        return x


class Encoder(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        dff,
        vocab_size,
        max_length,
        dropout_rate,
    ):
        super().__init__()

        self.pos_embedding = (
            PositionalEmbedding(
                vocab_size,
                d_model,
                max_length,
            )
        )

        self.dropout = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.layers_list = [
            EncoderLayer(
                d_model,
                num_heads,
                dff,
                dropout_rate,
            )
            for _
            in range(num_layers)
        ]

    def call(
        self,
        x,
        padding_mask,
        training=False,
    ):

        x = self.pos_embedding(x)

        x = self.dropout(
            x,
            training=training,
        )

        for layer in self.layers_list:

            x = layer(
                x,
                padding_mask=padding_mask,
                training=training,
            )

        return x

## Decoder

In [ ]:
# Decoder
class DecoderLayer(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        d_model,
        num_heads,
        dff,
        dropout_rate,
    ):
        super().__init__()

        self.causal_self_attention = (
            CausalSelfAttention(
                d_model,
                num_heads,
                dropout_rate,
            )
        )

        self.cross_attention = (
            CrossAttention(
                d_model,
                num_heads,
                dropout_rate,
            )
        )

        self.ffn = FeedForward(
            d_model,
            dff,
            dropout_rate,
        )

    def call(
        self,
        x,
        context,
        decoder_mask,
        context_mask,
        training=False,
    ):

        x = self.causal_self_attention(
            x,
            padding_mask=decoder_mask,
            training=training,
        )

        x = self.cross_attention(
            x,
            context,
            context_mask=context_mask,
            training=training,
        )

        x = self.ffn(
            x,
            training=training,
        )

        return x


class Decoder(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        dff,
        vocab_size,
        max_length,
        dropout_rate,
    ):
        super().__init__()

        self.pos_embedding = (
            PositionalEmbedding(
                vocab_size,
                d_model,
                max_length,
            )
        )

        self.dropout = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.layers_list = [
            DecoderLayer(
                d_model,
                num_heads,
                dff,
                dropout_rate,
            )
            for _
            in range(num_layers)
        ]

    def call(
        self,
        x,
        context,
        decoder_mask,
        context_mask,
        training=False,
    ):

        x = self.pos_embedding(x)

        x = self.dropout(
            x,
            training=training,
        )

        for layer in self.layers_list:

            x = layer(
                x,
                context,
                decoder_mask=decoder_mask,
                context_mask=context_mask,
                training=training,
            )

        return x

## Transformer

In [ ]:
# Transformer
# question -> Encoder -> Decoder -> logits
class Transformer(
    tf.keras.Model
):
    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        dff,
        vocab_size,
        max_length,
        dropout_rate=0.1,
    ):
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError(
                "d_model должен делиться "
                "на num_heads без остатка"
            )

        self.encoder = Encoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            vocab_size=vocab_size,
            max_length=max_length,
            dropout_rate=dropout_rate,
        )

        self.decoder = Decoder(
            num_layers=num_layers,
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            vocab_size=vocab_size,
            max_length=max_length,
            dropout_rate=dropout_rate,
        )

        self.final_layer = (
            tf.keras.layers.Dense(
                vocab_size
            )
        )

    def call(
        self,
        inputs,
        training=False,
    ):

        context_tokens, decoder_tokens = inputs

        # True = настоящий токен,
        # False = PAD.
        context_mask = tf.not_equal(
            context_tokens,
            PAD_ID,
        )

        decoder_mask = tf.not_equal(
            decoder_tokens,
            PAD_ID,
        )

        context = self.encoder(
            context_tokens,
            padding_mask=context_mask,
            training=training,
        )

        x = self.decoder(
            decoder_tokens,
            context,
            decoder_mask=decoder_mask,
            context_mask=context_mask,
            training=training,
        )

        logits = self.final_layer(x)

        return logits

## Loss и Accuracy

`[PAD]` используется только для выравнивания длины и не должен влиять
на оценку качества модели.

In [ ]:
# Loss и accuracy
loss_object = (
    tf.keras.losses
    .SparseCategoricalCrossentropy(
        from_logits=True,
        reduction="none",
    )
)


def masked_loss(
    labels,
    logits,
):

    loss = loss_object(
        labels,
        logits,
    )

    mask = tf.not_equal(
        labels,
        PAD_ID,
    )

    mask = tf.cast(
        mask,
        loss.dtype,
    )

    loss *= mask

    return (
        tf.reduce_sum(loss)
        / tf.reduce_sum(mask)
    )


def masked_accuracy(
    labels,
    logits,
):

    predicted = tf.argmax(
        logits,
        axis=-1,
        output_type=labels.dtype,
    )

    matches = tf.equal(
        labels,
        predicted,
    )

    mask = tf.not_equal(
        labels,
        PAD_ID,
    )

    matches = tf.logical_and(
        matches,
        mask,
    )

    matches = tf.cast(
        matches,
        tf.float32,
    )

    mask = tf.cast(
        mask,
        tf.float32,
    )

    return (
        tf.reduce_sum(matches)
        / tf.reduce_sum(mask)
    )

Warmup-график из оригинальной архитектуры Transformer.

In [ ]:
# Learning rate
class CustomSchedule(
    tf.keras.optimizers.schedules
    .LearningRateSchedule
):
    def __init__(
        self,
        d_model,
        warmup_steps=4000,
    ):
        super().__init__()

        self.d_model = tf.cast(
            d_model,
            tf.float32,
        )

        self.warmup_steps = (
            warmup_steps
        )

    def __call__(self, step):

        step = tf.cast(
            step,
            tf.float32,
        )

        arg1 = tf.math.rsqrt(
            tf.maximum(
                step,
                1.0,
            )
        )

        arg2 = (
            step
            * (
                self.warmup_steps
                ** -1.5
            )
        )

        return (
            tf.math.rsqrt(
                self.d_model
            )
            * tf.math.minimum(
                arg1,
                arg2,
            )
        )

Создание модели для автоматического сравнения нескольких наборов гиперпараметров.

In [ ]:
# Создание модели
def create_compiled_transformer(
    config,
):

    model = Transformer(
        num_layers=config[
            "num_layers"
        ],
        d_model=config[
            "d_model"
        ],
        num_heads=config[
            "num_heads"
        ],
        dff=config[
            "dff"
        ],
        vocab_size=(
            ACTUAL_VOCAB_SIZE
        ),
        max_length=MAX_LENGTH,
        dropout_rate=config[
            "dropout"
        ],
    )

    learning_rate = (
        CustomSchedule(
            config[
                "d_model"
            ],
            warmup_steps=config[
                "warmup_steps"
            ],
        )
    )

    optimizer = (
        tf.keras.optimizers.Adam(
            learning_rate,
            beta_1=0.9,
            beta_2=0.98,
            epsilon=1e-9,
        )
    )

    model.compile(
        optimizer=optimizer,
        loss=masked_loss,
        metrics=[
            masked_accuracy
        ],
    )

    return model

## Подбор гиперпараметров

Проверяем три архитектуры при одинаковой подвыборке данных.

Это небольшая учебная версия hyperparameter search.
Цель — не просто выбрать параметры вручную, а сравнить их по валидации.

Изменяем:

- число Encoder/Decoder слоёв;
- `d_model`;
- `num_heads`;
- размер Feed Forward;
- dropout.

In [ ]:
# Конфигурации
SEARCH_CONFIGS = [
    {
        "name": "small_4heads",
        "num_layers": 2,
        "d_model": 128,
        "num_heads": 4,
        "dff": 256,
        "dropout": 0.10,
        "warmup_steps": 1200,
    },

    {
        "name": "medium_6heads",
        "num_layers": 2,
        "d_model": 192,
        "num_heads": 6,
        "dff": 384,
        "dropout": 0.10,
        "warmup_steps": 1500,
    },

    {
        "name": "deeper_8heads",
        "num_layers": 3,
        "d_model": 192,
        "num_heads": 8,
        "dff": 512,
        "dropout": 0.15,
        "warmup_steps": 1500,
    },
]


SEARCH_TRAIN_N = min(
    16000,
    len(train_questions),
)

SEARCH_VAL_N = min(
    3000,
    len(val_questions),
)


search_train_ds = make_dataset(
    train_questions[
        :SEARCH_TRAIN_N
    ],
    train_answers[
        :SEARCH_TRAIN_N
    ],
    shuffle=True,
)

search_val_ds = make_dataset(
    val_questions[
        :SEARCH_VAL_N
    ],
    val_answers[
        :SEARCH_VAL_N
    ],
)


print(
    "Search train:",
    SEARCH_TRAIN_N,
)

print(
    "Search val:",
    SEARCH_VAL_N,
)

Каждая конфигурация обучается одинаковое число эпох на одинаковых данных.

In [ ]:
# Сравнение конфигураций
SEARCH_EPOCHS = 2

search_results = []


for config in SEARCH_CONFIGS:

    print(
        "\n"
        + "=" * 80
    )

    print(
        "CONFIG:",
        config["name"],
    )

    tf.keras.backend.clear_session()

    model = (
        create_compiled_transformer(
            config
        )
    )

    history = model.fit(
        search_train_ds,
        validation_data=search_val_ds,
        epochs=SEARCH_EPOCHS,
        verbose=2,
    )

    best_val_loss = min(
        history.history[
            "val_loss"
        ]
    )

    best_val_acc = max(
        history.history[
            "val_masked_accuracy"
        ]
    )

    search_results.append(
        {
            **config,
            "val_loss": (
                best_val_loss
            ),
            "val_masked_accuracy": (
                best_val_acc
            ),
        }
    )

    del model

    gc.collect()


search_df = (
    pd.DataFrame(
        search_results
    )
    .sort_values(
        "val_loss"
    )
    .reset_index(
        drop=True
    )
)


display(
    search_df
)

## Выбор лучшей конфигурации

In [ ]:
# Выбор лучшей конфигурации
best_row = (
    search_df
    .iloc[0]
)


best_config = {
    "name": best_row[
        "name"
    ],

    "num_layers": int(
        best_row[
            "num_layers"
        ]
    ),

    "d_model": int(
        best_row[
            "d_model"
        ]
    ),

    "num_heads": int(
        best_row[
            "num_heads"
        ]
    ),

    "dff": int(
        best_row[
            "dff"
        ]
    ),

    "dropout": float(
        best_row[
            "dropout"
        ]
    ),

    "warmup_steps": int(
        best_row[
            "warmup_steps"
        ]
    ),
}


print(
    "ЛУЧШАЯ КОНФИГУРАЦИЯ:"
)

print(
    best_config
)

## Обучение

Теперь создаём Transformer с лучшими найденными гиперпараметрами
и обучаем на всей тренировочной выборке.

Используем `EarlyStopping`, чтобы не продолжать обучение после ухудшения
валидационного loss.

In [ ]:
# Финальная модель
tf.keras.backend.clear_session()

final_model = (
    create_compiled_transformer(
        best_config
    )
)


# Один forward pass создаёт веса всех слоёв,
# после чего можно посмотреть summary.
sample_inputs, sample_targets = next(
    iter(train_ds)
)

_ = final_model(
    sample_inputs,
    training=False,
)


final_model.summary()

In [ ]:
# Обучение
FINAL_EPOCHS = 10


callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1,
    )
]


history = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINAL_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

Графики обучения

In [ ]:
# График обучения
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history.history[
        "loss"
    ],
    label="train loss",
)

plt.plot(
    history.history[
        "val_loss"
    ],
    label="val loss",
)

plt.xlabel(
    "Эпоха"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Transformer QA: loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# График обучения
plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history.history[
        "masked_accuracy"
    ],
    label="train accuracy",
)

plt.plot(
    history.history[
        "val_masked_accuracy"
    ],
    label="val accuracy",
)

plt.xlabel(
    "Эпоха"
)

plt.ylabel(
    "Masked accuracy"
)

plt.title(
    "Transformer QA: masked accuracy"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

Оценка на test

In [ ]:
# Оценка test
test_loss, test_accuracy = (
    final_model.evaluate(
        test_ds,
        verbose=0,
    )
)


print(
    f"TEST LOSS: {test_loss:.4f}"
)

print(
    "TEST MASKED ACCURACY: "
    f"{test_accuracy:.4f}"
)

## Генерация ответа

Во время inference правильный ответ модели уже неизвестен.

Поэтому:

1. Encoder получает вопрос один раз.
2. Decoder начинает с `[START]`.
3. Transformer предсказывает следующий токен.
4. Этот токен добавляется к ответу.
5. Цикл продолжается до `[END]` или достижения максимальной длины.

Используем greedy decoding (`argmax`).

In [ ]:
# Генерация ответа
def encode_question(
    question,
):

    question = clean_text(
        question
    )

    encoded = encode_text(
        question,
        max_length=MAX_LENGTH,
    )

    return np.asarray(
        [
            encoded
        ],
        dtype=np.int32,
    )


# question -> START -> tokens -> END
def generate_answer(
    question,
    model=final_model,
    max_answer_tokens=MAX_LENGTH - 1,
):

    encoder_input = (
        encode_question(
            question
        )
    )

    # Decoder начинает только с START.
    output_ids = [
        START_ID
    ]


    for _ in range(
        max_answer_tokens
    ):

        decoder_input = np.asarray(
            [
                output_ids
            ],
            dtype=np.int32,
        )


        logits = model(
            (
                encoder_input,
                decoder_input,
            ),
            training=False,
        )


        # Прогноз для последней позиции.
        next_token_logits = (
            logits[
                0,
                -1,
                :
            ]
        )


        next_id = int(
            tf.argmax(
                next_token_logits
            ).numpy()
        )


        if next_id == END_ID:
            break


        # PAD не должен становиться частью ответа.
        if next_id == PAD_ID:
            break


        output_ids.append(
            next_id
        )


    # START убираем.
    answer_ids = output_ids[
        1:
    ]


    answer = tokenizer.decode(
        answer_ids,
        skip_special_tokens=True,
    )


    return answer.strip()

Проверка чат-бота

In [ ]:
# Проверка чат-бота
CHAT_QUESTIONS = [
    "привет, как у тебя дела?",

    "что делаешь сегодня вечером?",

    "как настроение?",

    "ты любишь музыку?",

    "что посоветуешь посмотреть вечером?",

    "как лучше провести выходные?",

    "почему люди иногда грустят?",

    "ты умеешь готовить?",

    "расскажи что-нибудь интересное",

    "как познакомиться с новым человеком?",
]


chat_results = []


for question in CHAT_QUESTIONS:

    answer = generate_answer(
        question
    )

    chat_results.append(
        {
            "question": question,
            "answer": answer,
        }
    )


chat_df = pd.DataFrame(
    chat_results
)


display(
    chat_df
)

## Сравнение с эталоном

Теперь берём несколько реальных вопросов из test и выводим:

- вопрос;
- эталон из датасета;
- ответ Transformer.

In [ ]:
# Сравнение с эталоном
SAMPLE_N = 15


sample_test = test_df.sample(
    n=min(
        SAMPLE_N,
        len(test_df),
    ),
    random_state=SEED,
)


comparison_rows = []


for _, row in sample_test.iterrows():

    prediction = (
        generate_answer(
            row["question"]
        )
    )

    comparison_rows.append(
        {
            "question": (
                row["question"]
            ),

            "reference": (
                row["answer"]
            ),

            "prediction": (
                prediction
            ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


display(
    comparison_df
)

## Метрика chrF

Для генеративной задачи обычная accuracy недостаточна.

Один и тот же смысл можно выразить разными словами.

Поэтому дополнительно считаем **chrF** — метрику сходства текста по
символьным n-граммам.

In [ ]:
# Метрика chrF
EVAL_GENERATION_N = 50


generation_test = test_df.sample(
    n=min(
        EVAL_GENERATION_N,
        len(test_df),
    ),
    random_state=123,
)


chrf_scores = []


for index, row in generation_test.iterrows():

    prediction = generate_answer(
        row["question"]
    )

    reference = row["answer"]


    score = (
        sacrebleu
        .sentence_chrf(
            prediction,
            [
                reference
            ],
        )
        .score
    )


    chrf_scores.append(
        score
    )


mean_chrf = float(
    np.mean(
        chrf_scores
    )
)


print(
    "Средний chrF на "
    f"{len(chrf_scores)} примерах: "
    f"{mean_chrf:.2f}"
)

Сохранение модели. Keras-модель сохраняем отдельно, а словарь WordPiece — в текстовый файл.

In [ ]:
# Сохранение модели
MODEL_PATH = (
    "/content/"
    "russian_dialogues_transformer.keras"
)


final_model.save(
    MODEL_PATH
)


TOKENIZER_SAVE_DIR = Path(
    "/content/"
    "russian_dialogue_wordpiece"
)


TOKENIZER_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


tokenizer.save_model(
    str(
        TOKENIZER_SAVE_DIR
    )
)


print(
    "Модель:",
    MODEL_PATH
)

print(
    "Tokenizer:",
    TOKENIZER_SAVE_DIR
)